# Self-Healing Feeder: Active-Grid Transfer to Urban Underground IEEE 37-bus (4.8 kV, 3-wire delta)

Runs the active-grid and Split Conformal Prediction sections of `research/ieee37-underground/PLAN.md` on Kaggle GPU
(offline OpenDSS wheels + PyTorch CUDA GPU):
1. Active OpenDSS sweeps on IEEE 123-bus (training) and unseen urban underground IEEE 37-bus (3-wire ungrounded delta,
   0-50% delta-connected `Model=7` DERs, `normal` / `reconfig_tie` / `looped_tie` via `Tie37`, 4 kV resistance grid,
   and 50/50 `pv_cloud` / `load_step` `NO_FAULT` events with zero artificial capacitors).
2. Trains `ActivePerNodeGBDT` and 3-layer bidirectional `ActiveGridGATv2` strictly on IEEE 123-bus (5-fold GroupKFold by bus).
3. Evaluates `active_fpi`, zero-shot transfer, in-domain 5-fold CV upper bound, and 5%/20% few-shot on IEEE 37-bus.
4. Runs Split Conformal Prediction (`aps_det`, `lac`, and pre-registered `aps_rand` at 95%) plus calibration-size sweep
   (5%-40%, 50 random draws).
5. Writes isolated outputs to `results/active_grid_ieee37/` + `data/generated/active_grid_ieee37_probs.npz` + provenance
   (never overwrites `results/active_grid/`).

Simulation only. Nothing here is field data.

In [ ]:
import os, shutil, zipfile

DATASET_DIR = "/kaggle/input"
WORK = "/kaggle/working/self-healing-feeder"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

src = next((root for root, _dirs, files in os.walk(DATASET_DIR) if "run.py" in files), None)
if src is None:
    zips = [os.path.join(r, f) for r, _d, fs in os.walk(DATASET_DIR) for f in fs if f.endswith(".zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/kaggle/working/_bundle")
    src = next((root for root, _dirs, files in os.walk("/kaggle/working/_bundle") if "run.py" in files), None)
if src is None:
    raise SystemExit("Could not find project bundle in /kaggle/input")
shutil.copytree(src, WORK)
shutil.rmtree("/kaggle/working/_bundle", ignore_errors=True)
os.chdir(WORK)
print("Working dir:", WORK)
print("Top-level:", sorted(os.listdir(WORK)))

In [ ]:
# Install OpenDSS offline from bundled Linux wheels (no Internet required).
import subprocess, sys
if os.path.isdir(os.path.join(WORK, "wheels")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                    "--find-links", "wheels", "OpenDSSDirect.py==0.9.4"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "OpenDSSDirect.py==0.9.4"], check=True)

import torch, opendssdirect as dss
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("OpenDSS:", dss.Basic.Version())

In [ ]:
# Fast verification tests on Kaggle environment before running the full experiment.
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_active_grid.py"], check=True)

In [ ]:
# Run the full Active-Grid GNN + Split Conformal experiment for unseen IEEE 37-bus.
subprocess.run([sys.executable, "-u", "scripts/active_grid_run.py", "--test-feeder", "ieee37"], check=True)

In [ ]:
# Pre-registered randomized APS (aps_rand at 95%) comparison + calibration-size sweep (5%-40%, 50 draws).
subprocess.run([sys.executable, "-u", "scripts/active_conformal_posthoc.py", "--test-feeder", "ieee37"], check=True)
subprocess.run([sys.executable, "-u", "scripts/active_conformal_calsize.py", "--test-feeder", "ieee37"], check=True)

In [ ]:
# Package results/active_grid_ieee37, provenance records, and probabilities into /kaggle/working.
import json
from pathlib import Path

OUT = Path("/kaggle/working/ieee37_active_results")
if OUT.exists():
    shutil.rmtree(OUT)
(OUT / "active_grid_ieee37").mkdir(parents=True)
(OUT / "provenance").mkdir(parents=True)
(OUT / "generated").mkdir(parents=True)

for p in sorted((Path(WORK) / "results" / "active_grid_ieee37").glob("*.json")):
    shutil.copy2(p, OUT / "active_grid_ieee37" / p.name)
for prod in ("active_grid_ieee37", "active_conformal_posthoc_ieee37", "active_conformal_calsize_ieee37"):
    shutil.copy2(Path(WORK) / "results" / "provenance" / f"{prod}.json", OUT / "provenance" / f"{prod}.json")
shutil.copy2(Path(WORK) / "data" / "generated" / "active_grid_ieee37_probs.npz",
             OUT / "generated" / "active_grid_ieee37_probs.npz")

os.chdir("/kaggle/working")
shutil.rmtree(WORK)
shutil.make_archive("/kaggle/working/ieee37_active_results", "zip", OUT)

p37 = json.loads((OUT / "active_grid_ieee37" / "phase_active_ieee37.json").read_text())
print("=== Pre-committed criteria summary (IEEE 37-bus active) ===")
print(json.dumps(p37["pre_committed_criteria"], indent=2))
print("Saved /kaggle/working/ieee37_active_results.zip")